In [48]:
# SETUP AND LOAD DATA
from pathlib import Path
import pandas as pd
import numpy as np

AIRBNB_DIR = Path("../Airbnb")

raw_files = list(AIRBNB_DIR.glob("Airbnb_NYC_2019_RAW*.csv"))

if not raw_files:
    raise FileNotFoundError(
        f"No raw Airbnb CSV found in: {AIRBNB_DIR}"
    )

RAW_FILE = raw_files[0]

print("Using dataset:")
print(RAW_FILE)

df = pd.read_csv(RAW_FILE)

df.head()

Using dataset:
../Airbnb/Airbnb_NYC_2019_RAW.csv


/var/folders/8v/frb1qkxn6tvb4c7lz7kfr90r0000gn/T/ipykernel_11925/2132463042.py:20: DtypeWarning: Columns (14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(RAW_FILE)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [49]:
# Create copy for cleaning
clean_df = df.copy()

In [50]:
#calculated_host_listings_count was stored as object instead of numeric. 
clean_df["calculated_host_listings_count"] = pd.to_numeric(
    clean_df["calculated_host_listings_count"],
    errors="coerce"
)

In [51]:
clean_df.dtypes


#VERIFIED

id                                  int64
name                               object
host_id                             int64
host_name                          object
neighbourhood_group                object
neighbourhood                      object
latitude                          float64
longitude                         float64
room_type                          object
price                               int64
minimum_nights                      int64
number_of_reviews                   int64
last_review                        object
reviews_per_month                 float64
calculated_host_listings_count    float64
availability_365                    int64
dtype: object

Step 3 — Handle Missing Values
3A — Missing host_name

In [52]:
print(clean_df["host_name"].isnull().sum())

21


Cleaning Decision
Replace with "Unknown Host".

In [53]:
clean_df["host_name"] = clean_df["host_name"].fillna("Unknown Host")

In [54]:
print(clean_df["host_name"].isnull().sum())

0


3B — Missing name
Cleaning Decision
Replace with "Unknown Listing".

In [55]:
clean_df["name"] = clean_df["name"].fillna("Unknown Listing")

3C — Convert Date Column Properly

In [56]:
clean_df["last_review"] = pd.to_datetime(
    clean_df["last_review"],
    errors="coerce"
)

3D — Handle Missing reviews_per_month
This is an important logical cleaning step.
Logic
If:
number_of_reviews == 0
and reviews_per_month is missing
then replace with 0.

In [57]:
clean_df.loc[
    (clean_df["number_of_reviews"] == 0) &
    (clean_df["reviews_per_month"].isnull()),
    "reviews_per_month"
] = 0

In [58]:
print(clean_df["reviews_per_month"].isnull().sum())

0


Remove Invalid Zero Prices

In [59]:
print((clean_df["price"] <= 0).sum())

11


In [60]:
clean_df = clean_df[clean_df["price"] > 0]

Step 5 — Cap Extreme Price Outliers
Use IQR method.
Calculate Limits

In [61]:
# Calculate limit
Q1 = clean_df["price"].quantile(0.25)
Q3 = clean_df["price"].quantile(0.75)

IQR = Q3 - Q1

upper_limit = Q3 + (1.5 * IQR)

print("Upper limit:", upper_limit)

Upper limit: 334.0


In [62]:
# / remove extreme outliers
clean_df = clean_df[
    clean_df["price"] <= upper_limit
]

Step 6 — Clean Minimum Nights
Very high values are unrealistic.
Inspect

In [63]:
print(clean_df["minimum_nights"].max())

1250


In [64]:

clean_df = clean_df[
    clean_df["minimum_nights"] <= 365
]


# You can justify:
# values > 365 are operationally unrealistic for Airbnb-style rentals.

Step 7 — Standardise Text Columns
Remove whitespace inconsistencies.

In [65]:
text_columns = [
    "name",
    "host_name",
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

for col in text_columns:
    clean_df[col] = clean_df[col].str.strip()

In [66]:
clean_df.info()

clean_df.isnull().sum()

clean_df.describe()

<class 'pandas.core.frame.DataFrame'>
Index: 45899 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              45899 non-null  int64         
 1   name                            45899 non-null  object        
 2   host_id                         45899 non-null  int64         
 3   host_name                       45899 non-null  object        
 4   neighbourhood_group             45899 non-null  object        
 5   neighbourhood                   45899 non-null  object        
 6   latitude                        45899 non-null  float64       
 7   longitude                       45899 non-null  float64       
 8   room_type                       45899 non-null  object        
 9   price                           45899 non-null  int64         
 10  minimum_nights                  45899 non-null  int64         
 11  number_

,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
count,4.589900e+04,4.589900e+04,45899.000000,45899.000000,45899.000000,45899.000000,45899.000000,36895,45899.000000,45898.00000,45899.000000
mean,1.889952e+07,6.633540e+07,40.728491,-73.950737,119.999804,6.757729,23.945358,2018-10-02 18:58:46.136332800,1.107894,6.64188,109.326042
min,2.539000e+03,2.438000e+03,40.499790,-74.244420,10.000000,1.000000,0.000000,2011-03-28 00:00:00,0.000000,1.00000,0.000000
25%,9.436180e+06,7.720804e+06,40.689230,-73.981940,65.000000,1.000000,1.000000,2018-07-02 12:00:00,0.050000,1.00000,0.000000
50%,1.952538e+07,3.028459e+07,40.721780,-73.954370,100.000000,2.000000,5.000000,2019-05-19 00:00:00,0.380000,1.00000,39.000000
75%,2.891372e+07,1.055431e+08,40.763405,-73.934325,159.000000,5.000000,24.000000,2019-06-23 00:00:00,1.620000,2.00000,216.000000
max,3.648724e+07,2.743213e+08,40.913060,-73.712990,334.000000,365.000000,629.000000,2019-07-08 00:00:00,58.500000,327.00000,365.000000
std,1.092027e+07,7.756683e+07,0.055335,0.046478,68.135132,16.002435,45.321310,NaN,1.612729,31.01641,130.253339


In [67]:
# Save cleaned dataset before one-hot encoding

CLEANED_FILE = AIRBNB_DIR / "AB_NYC_2019_Cleaned_Final.csv"

clean_df.to_csv(
    CLEANED_FILE,
    index=False
)

print(f"Cleaned dataset saved to: {CLEANED_FILE}")

Cleaned dataset saved to: ../Airbnb/AB_NYC_2019_Cleaned_Final.csv


In [68]:
# ------------------------------------------------------------
# ENCODING CATEGORICAL VARIABLES
# ------------------------------------------------------------

# Create a copy of cleaned dataset
encoded_df = clean_df.copy()

# Select categorical columns for encoding
categorical_cols = [
    "room_type",
    "neighbourhood_group"
]

# Apply One-Hot Encoding
encoded_df = pd.get_dummies(
    encoded_df,
    columns=categorical_cols,
    drop_first=False
)

# ------------------------------------------------------------
# VIEW ENCODED DATASET
# ------------------------------------------------------------

print("Encoded Dataset Shape:")
print(encoded_df.shape)

print("\nEncoded Columns:")
print(encoded_df.columns)

# Show first rows
encoded_df.head()

Encoded Dataset Shape:
(45899, 22)

Encoded Columns:
Index(['id', 'name', 'host_id', 'host_name', 'neighbourhood', 'latitude',
       'longitude', 'price', 'minimum_nights', 'number_of_reviews',
       'last_review', 'reviews_per_month', 'calculated_host_listings_count',
       'availability_365', 'room_type_Entire home/apt',
       'room_type_Private room', 'room_type_Shared room',
       'neighbourhood_group_Bronx', 'neighbourhood_group_Brooklyn',
       'neighbourhood_group_Manhattan', 'neighbourhood_group_Queens',
       'neighbourhood_group_Staten Island'],
      dtype='object')


,id,name,host_id,host_name,neighbourhood,latitude,longitude,price,minimum_nights,number_of_reviews,...,calculated_host_listings_count,availability_365,room_type_Entire home/apt,room_type_Private room,room_type_Shared room,neighbourhood_group_Bronx,neighbourhood_group_Brooklyn,neighbourhood_group_Manhattan,neighbourhood_group_Queens,neighbourhood_group_Staten Island
0,2539,Clean & quiet apt home by the park,2787,John,Kensington,40.64749,-73.97237,149,1,9,...,6.0,365,False,True,False,False,True,False,False,False
1,2595,Skylit Midtown Castle,2845,Jennifer,Midtown,40.75362,-73.98377,225,1,45,...,2.0,355,True,False,False,False,False,True,False,False
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Harlem,40.80902,-73.94190,150,3,0,...,1.0,365,False,True,False,False,False,True,False,False
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Clinton Hill,40.68514,-73.95976,89,1,270,...,1.0,194,True,False,False,False,True,False,False,False
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,East Harlem,40.79851,-73.94399,80,10,9,...,1.0,0,True,False,False,False,False,True,False,False


In [69]:
# Save encoded dataset separately

ENCODED_FILE = AIRBNB_DIR / "AB_NYC_2019_Encoded.csv"

encoded_df.to_csv(
    ENCODED_FILE,
    index=False
)

print(f"Encoded dataset saved to: {ENCODED_FILE}")

Encoded dataset saved to: ../Airbnb/AB_NYC_2019_Encoded.csv
